In [1]:
"""
@author: Zilan Cheng
@note: This code is modified based on Zongyi Li's original implementation of Fourier Neural Operators.
"""

import torch.nn.functional as F
from timeit import default_timer
from utilities3 import *
torch.cuda.set_device(0)
torch.manual_seed(0)
np.random.seed(0)

In [2]:
class SpectralConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super(SpectralConv2d, self).__init__()
        """
        2D Fourier layer. It does FFT, linear transform, and Inverse FFT.    
        """
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1 
        self.modes2 = modes2

        self.scale = (1 / (in_channels * out_channels))
        self.weights1 = nn.Parameter(self.scale * torch.rand(self.in_channels, self.out_channels, self.modes1, self.modes2, dtype=torch.cfloat))
        self.weights2 = nn.Parameter(self.scale * torch.rand(self.in_channels, self.out_channels, self.modes1, self.modes2, dtype=torch.cfloat))

    def compl_mul2d(self, input, weights):
        return torch.einsum("bixy,ioxy->boxy", input, weights)

    def forward(self, x):
        batchsize = x.shape[0]
        x_ft = torch.fft.rfft2(x)

        out_ft = torch.zeros(batchsize, self.out_channels,  x.size(-2), x.size(-1)//2 + 1, dtype=torch.cfloat, device=x.device)
        out_ft[:, :, :self.modes1, :self.modes2] = \
            self.compl_mul2d(x_ft[:, :, :self.modes1, :self.modes2], self.weights1)
        out_ft[:, :, -self.modes1:, :self.modes2] = \
            self.compl_mul2d(x_ft[:, :, -self.modes1:, :self.modes2], self.weights2)

        x = torch.fft.irfft2(out_ft, s=(x.size(-2), x.size(-1)))
        return x

In [3]:
class MLP(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels):
        super(MLP, self).__init__()
        self.mlp1 = nn.Conv2d(in_channels, mid_channels, 1)
        self.mlp2 = nn.Conv2d(mid_channels, out_channels, 1)

    def forward(self, x):
        x = self.mlp1(x)
        x = F.gelu(x)
        x = self.mlp2(x)
        return x

In [4]:
def get_grid(shape, device):
    batchsize, size_x, size_y = shape[0], shape[1], shape[2]
    gridx = torch.tensor(np.linspace(0, 1, size_x), dtype=torch.float)
    gridx = gridx.reshape(1, size_x, 1, 1).repeat([batchsize, 1, size_y, 1])
    gridy = torch.tensor(np.linspace(0, 1, size_y), dtype=torch.float)
    gridy = gridy.reshape(1, 1, size_y, 1).repeat([batchsize, size_x, 1, 1])
    return torch.cat((gridx, gridy), dim=-1).to(device)

In [5]:
class FNO2d(nn.Module):
    def __init__(self, modes1, modes2,  width):
        super(FNO2d, self).__init__()

        self.modes1 = modes1
        self.modes2 = modes2
        self.width = width
        self.padding = 9 # pad the domain if input is non-periodic

        self.p = nn.Linear(3, self.width) # input channel is 3: (a(x, y), x, y)
        self.conv0 = SpectralConv2d(self.width, self.width, self.modes1, self.modes2)
        self.conv1 = SpectralConv2d(self.width, self.width, self.modes1, self.modes2)
        self.conv2 = SpectralConv2d(self.width, self.width, self.modes1, self.modes2)
        self.conv3 = SpectralConv2d(self.width, self.width, self.modes1, self.modes2)
        self.mlp0 = MLP(self.width, self.width, self.width)
        self.mlp1 = MLP(self.width, self.width, self.width)
        self.mlp2 = MLP(self.width, self.width, self.width)
        self.mlp3 = MLP(self.width, self.width, self.width)
        self.w0 = nn.Conv2d(self.width, self.width, 1)
        self.w1 = nn.Conv2d(self.width, self.width, 1)
        self.w2 = nn.Conv2d(self.width, self.width, 1)
        self.w3 = nn.Conv2d(self.width, self.width, 1)
        self.q = MLP(self.width, 1, self.width * 4) # output channel is 1: u(x, y)

    def forward(self, x):
        grid = get_grid(x.shape, x.device)
        x = torch.cat((x, grid), dim=-1)
        x = self.p(x)
        x = x.permute(0, 3, 1, 2)
        x = F.pad(x, [0,self.padding, 0,self.padding])

        x1 = self.conv0(x)
        x1 = self.mlp0(x1)
        x2 = self.w0(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv1(x)
        x1 = self.mlp1(x1)
        x2 = self.w1(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv2(x)
        x1 = self.mlp2(x1)
        x2 = self.w2(x)
        x = x1 + x2
        x = F.gelu(x)

        x1 = self.conv3(x)
        x1 = self.mlp3(x1)
        x2 = self.w3(x)
        x = x1 + x2

        x = x[..., :-self.padding, :-self.padding]
        x = self.q(x)
        x = x.permute(0, 2, 3, 1)
        return x

In [6]:
################################################################
# configs
################################################################
ntrain = 900
ntest = 100

modes = 6
width = 32

r = 5
h = int(((421 - 1)/r) + 1)
s = h

batch_size = 20
learning_rate = 0.001
epochs = 1000
iterations = epochs*(ntrain//batch_size)

In [7]:
################################################################
# dataloader and data normalization
################################################################
TRAIN_PATH = '../data/darcy/piececonst_r421_N1024_smooth1.mat'
TEST_PATH = '../data/darcy/piececonst_r421_N1024_smooth2.mat'

reader = MatReader(TRAIN_PATH)
x_train = reader.read_field('coeff')[:ntrain,::r,::r][:,:s,:s]
y_train = reader.read_field('sol')[:ntrain,::r,::r][:,:s,:s]

reader.load_file(TEST_PATH)
x_test = reader.read_field('coeff')[:ntest,::r,::r][:,:s,:s]
y_test = reader.read_field('sol')[:ntest,::r,::r][:,:s,:s]

x_train = x_train.reshape(ntrain,s,s,1)
x_test = x_test.reshape(ntest,s,s,1)
y_train = y_train.reshape(ntrain,s,s,1)
y_test = y_test.reshape(ntest,s,s,1)

x_normalizer = UnitGaussianNormalizer(x_train)
x_train = x_normalizer.encode(x_train)
x_test = x_normalizer.encode(x_test)

y_normalizer = UnitGaussianNormalizer(y_train)
y_train = y_normalizer.encode(y_train)

train_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x_test, y_test), batch_size=batch_size, shuffle=False)

In [8]:
solution_real=torch.zeros(ntest,s,s,1)
solution_learnt=torch.zeros(ntest,s,s,1)

In [9]:
################################################################
# training and evaluation
################################################################
model = FNO2d(modes, modes, width).to(device) 
print(count_params(model))

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=iterations)

myloss = LpLoss(size_average=True)
y_normalizer.to(device)
for ep in range(epochs):
    i=0
    model.train()
    train_l2 = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x).reshape(batch_size, s, s,1)
        out = y_normalizer.decode(out)
        y = y_normalizer.decode(y)

        loss = myloss(out.view(batch_size,-1), y.view(batch_size,-1))
        loss.backward()

        optimizer.step()
        scheduler.step()
        train_l2 += loss.item()

    model.eval()
    test_l2 = 0.0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)

            out = model(x).reshape(batch_size, s, s,1)
            out = y_normalizer.decode(out)
            solution_real[i:i+batch_size,:,:,:]=y
            solution_learnt[i:i+batch_size,:,:,:]=out
            i=i+batch_size
            test_l2 += myloss(out.view(batch_size,-1), y.view(batch_size,-1)).item()
    train_l2/= ntrain/batch_size
    test_l2 /= ntest/batch_size

    if ep % 50 == 0 or ep == epochs - 1:
        print(f"Epoch {ep:4d} | Train L2: {train_l2:.6f} | Test L2: {test_l2:.6f}")


606977
Epoch    0 | Train L2: 0.190150 | Test L2: 0.123495
Epoch   50 | Train L2: 0.025101 | Test L2: 0.027926
Epoch  100 | Train L2: 0.022013 | Test L2: 0.022210
Epoch  150 | Train L2: 0.019311 | Test L2: 0.022581
Epoch  200 | Train L2: 0.018109 | Test L2: 0.018390
Epoch  250 | Train L2: 0.016966 | Test L2: 0.018394
Epoch  300 | Train L2: 0.016809 | Test L2: 0.016948
Epoch  350 | Train L2: 0.015960 | Test L2: 0.016106
Epoch  400 | Train L2: 0.015972 | Test L2: 0.015399
Epoch  450 | Train L2: 0.015259 | Test L2: 0.016073
Epoch  500 | Train L2: 0.013713 | Test L2: 0.014190
Epoch  550 | Train L2: 0.013050 | Test L2: 0.014744
Epoch  600 | Train L2: 0.012630 | Test L2: 0.013631
Epoch  650 | Train L2: 0.012487 | Test L2: 0.013273
Epoch  700 | Train L2: 0.012064 | Test L2: 0.013732
Epoch  750 | Train L2: 0.011590 | Test L2: 0.012790
Epoch  800 | Train L2: 0.011494 | Test L2: 0.012884
Epoch  850 | Train L2: 0.011227 | Test L2: 0.012432
Epoch  900 | Train L2: 0.011157 | Test L2: 0.012326
Epoch